# GeoTIFF Pipeline: NetCDF → PostgreSQL Ingestion (Multi-Dataset)

Ingests NetCDF files into the `climate_gt` table with per-dataset tracking via the `datasets` table.

Each NetCDF file gets a unique UUID (`datasets.id`) and a user-provided name (`datasets.name`).

Idempotent: `UNIQUE (dataset_id, time, lat, lon, variable)` prevents duplicates.

## Configuration

In [ ]:
from pathlib import Path
import psycopg2
import psycopg2.extras
import numpy as np
import sys
import xarray as xr
import pandas as pd

# ── User Configuration ─────────────────────────────────────────────────
# Set these before running:
DATASET_NAME = "rainfall_1989"                       # Unique human-readable name
NC_FILE = Path("../local_only/1989.monthly_rain.nc") # Path to NetCDF file
# ────────────────────────────────────────────────────────────────────────

# Database connection parameters
DB_PARAMS = {
    'host': 'localhost',
    'user': 'leon',
    'password': 'leon',
    'database': 'oc-database',
    'port': 5432
}

print(f"Dataset name: {DATASET_NAME}")
print(f"NetCDF file:  {NC_FILE}")

## Create Tables

In [ ]:
conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

cur.execute("""
    CREATE EXTENSION IF NOT EXISTS "pgcrypto";

    CREATE TABLE IF NOT EXISTS datasets (
        id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
        name TEXT UNIQUE NOT NULL,
        filename TEXT,
        created_at TIMESTAMP DEFAULT NOW(),
        metadata JSONB
    );

    CREATE TABLE IF NOT EXISTS climate_gt (
        dataset_id UUID REFERENCES datasets(id) ON DELETE CASCADE,
        time TIMESTAMP,
        lat DOUBLE PRECISION,
        lon DOUBLE PRECISION,
        variable TEXT,
        value DOUBLE PRECISION,
        UNIQUE (dataset_id, time, lat, lon, variable)
    );

    CREATE INDEX IF NOT EXISTS idx_climate_gt_dataset_id ON climate_gt (dataset_id);
    CREATE INDEX IF NOT EXISTS idx_climate_gt_variable ON climate_gt (variable);
    CREATE INDEX IF NOT EXISTS idx_climate_gt_time ON climate_gt (time);
""")

conn.commit()
cur.close()
conn.close()

print("Tables created (datasets, climate_gt)")

## Register Dataset

In [ ]:
conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

cur.execute("""
    INSERT INTO datasets (name, filename)
    VALUES (%s, %s)
    ON CONFLICT (name) DO UPDATE SET filename = EXCLUDED.filename
    RETURNING id
""", (DATASET_NAME, NC_FILE.name))

DATASET_ID = cur.fetchone()[0]
conn.commit()
cur.close()
conn.close()

print(f"Dataset registered: {DATASET_NAME}")
print(f"Dataset ID (UUID): {DATASET_ID}")

## NetCDFViewer Class

In [ ]:
class NetCDFViewer:
    """Context manager for exploring and extracting data from NetCDF files."""
    
    def __init__(self, path):
        self.path = Path(path)
        self.ds = None
    
    def __enter__(self):
        self.ds = xr.open_dataset(self.path, engine="netcdf4")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.ds is not None:
            self.ds.close()
        return False
    
    def get_spatial_slice(self, time, variable=None):
        """Extract a spatial slice for a given time step."""
        if self.ds is None:
            raise RuntimeError("Dataset not open. Use 'with NetCDFViewer(path) as viewer:'")
        
        if variable is None:
            numeric_vars = [
                name for name, var in self.ds.data_vars.items()
                if var.dtype not in [object, str] and len(var.dims) > 0
            ]
            if len(numeric_vars) == 0:
                raise ValueError("No numeric data variables found.")
            elif len(numeric_vars) == 1:
                variable = numeric_vars[0]
            else:
                raise ValueError(f"Multiple data variables found: {numeric_vars}. Specify one.")
        
        return self.ds[variable].sel(time=time, method="nearest")
    
    def get_all_times(self):
        """Return all time coordinates."""
        if self.ds is None:
            raise RuntimeError("Dataset not open.")
        return self.ds.coords['time'].values

## Ingestion Function

In [ ]:
def ingest_netcdf_file(nc_file_path, db_params, dataset_id, variable_name=None, batch_size=10000):
    """
    Ingest a NetCDF file into climate_gt, linked to dataset_id.
    """
    conn = psycopg2.connect(**db_params)
    cur = conn.cursor()
    
    try:
        with NetCDFViewer(nc_file_path) as viewer:
            times = viewer.get_all_times()
            print(f"Found {len(times)} time steps")
            
            if variable_name is None:
                with xr.open_dataset(nc_file_path, engine="netcdf4") as ds:
                    numeric_vars = [
                        name for name, var in ds.data_vars.items()
                        if var.dtype not in [object, str] and len(var.dims) > 0
                    ]
                    if len(numeric_vars) == 1:
                        variable_name = numeric_vars[0]
                    else:
                        raise ValueError(f"Could not auto-detect variable. Found: {numeric_vars}")
            
            print(f"Ingesting variable: {variable_name}")
            
            total_rows = 0
            for time_idx, time_val in enumerate(times):
                spatial_slice = viewer.get_spatial_slice(
                    pd.Timestamp(time_val),
                    variable=variable_name
                )
                
                lats = spatial_slice.coords['lat'].values
                lons = spatial_slice.coords['lon'].values
                
                rows = []
                time_py = pd.Timestamp(time_val).to_pydatetime()
                for i, lat in enumerate(lats):
                    for j, lon in enumerate(lons):
                        value = spatial_slice.values[i, j]
                        if not np.isnan(value):
                            rows.append((str(dataset_id), time_py, float(lat), float(lon), variable_name, float(value)))
                
                print(f"  Time {time_idx + 1}/{len(times)}: {time_val} -> {len(rows):,} rows")
                total_rows += len(rows)
                
                if rows:
                    psycopg2.extras.execute_values(
                        cur,
                        """INSERT INTO climate_gt (dataset_id, time, lat, lon, variable, value)
                           VALUES %s
                           ON CONFLICT DO NOTHING""",
                        rows,
                        page_size=batch_size
                    )
        
        # Update dataset metadata w/ summary
        cur.execute("""
            UPDATE datasets SET metadata = %s::jsonb WHERE id = %s
        """, (
            psycopg2.extras.Json({
                'variable': variable_name,
                'time_steps': len(times),
                'total_rows': total_rows,
            }),
            str(dataset_id)
        ))
        
        conn.commit()
        print(f"\nIngestion complete! {total_rows:,} total rows")
        
    except Exception as e:
        conn.rollback()
        print(f"Error during ingestion: {e}")
        raise
    finally:
        cur.close()
        conn.close()

## Run Ingestion

In [ ]:
ingest_netcdf_file(NC_FILE, DB_PARAMS, DATASET_ID, batch_size=10000)

## Verify Ingestion

In [ ]:
conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

# List all datasets
cur.execute("SELECT id, name, filename, created_at FROM datasets ORDER BY created_at")
print("All datasets:")
for row in cur.fetchall():
    print(f"  {row[0]} | {row[1]} | {row[2]} | {row[3]}")

# Row count for current dataset
cur.execute("SELECT COUNT(*) FROM climate_gt WHERE dataset_id = %s", (str(DATASET_ID),))
total_rows = cur.fetchone()[0]
print(f"\nRows in climate_gt for '{DATASET_NAME}': {total_rows:,}")

# Variables for current dataset
cur.execute("""
    SELECT variable, COUNT(*) as row_count
    FROM climate_gt
    WHERE dataset_id = %s
    GROUP BY variable
""", (str(DATASET_ID),))
print("\nRows per variable:")
for var, count in cur.fetchall():
    print(f"  {var}: {count:,}")

# Unique time steps
cur.execute("""
    SELECT COUNT(DISTINCT time) FROM climate_gt WHERE dataset_id = %s
""", (str(DATASET_ID),))
print(f"\nUnique time steps: {cur.fetchone()[0]}")

# Sample rows
cur.execute("""
    SELECT time, lat, lon, variable, value
    FROM climate_gt
    WHERE dataset_id = %s
    LIMIT 5
""", (str(DATASET_ID),))
print("\nSample rows:")
for row in cur.fetchall():
    print(f"  {row}")

cur.close()
conn.close()